# End-to-End Walkthrough — Supply Chain Analytics
This notebook walks the full pipeline interactively: ingestion, augmentation,
quality gates, EDA, forecasting, and anomaly validation.

**Prerequisite:** place `DataCoSupplyChainDataset.csv` in `data/raw/`
(see `data/raw/README.md`). Without it, the pipeline uses the 5k dev sample
and all metrics are smoke-test only.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '../etl')
import data_ingestion, augment_data
std = data_ingestion.main()
augment_data.main()

## 1. EDA — what does the real data look like?

In [ ]:
tx = pd.read_csv('../data/processed/transactions_augmented.csv',
                 parse_dates=['order_date'])
print(tx.shape)
tx[['sales','quantity','product_price','ship_days_actual']].describe().round(2)

In [ ]:
# Monthly revenue + OTIF trend
tx['on_time'] = tx['ship_days_actual'] <= tx['ship_days_scheduled']
m = tx.set_index('order_date').resample('MS').agg(
    revenue=('sales','sum'), otif=('on_time','mean'))
fig, ax = plt.subplots(1, 2, figsize=(13,4))
m['revenue'].plot(ax=ax[0], title='Monthly Revenue')
(m['otif']*100).plot(ax=ax[1], title='OTIF %', color='darkorange')
plt.tight_layout()

In [ ]:
# Category revenue vs profitability
cat = tx.groupby('category_name').agg(
    revenue=('sales','sum'), profit=('order_profit','sum')).sort_values('revenue')
cat.plot.barh(figsize=(9,5), title='Revenue vs Profit by Category');

## 2. Quality gates
Every run appends to `data/quality_audit/quality_audit_log.csv` — the audit trail.

In [ ]:
from data_quality_framework import DataQualityGates
report = DataQualityGates(tx, source='notebook_run').run_all()
report[['check','dimension','failed_rows','failure_rate','status']]

## 3. Demand forecasting (walk-forward backtest)
Headline metric is **WMAPE** (robust to low-volume weeks).
Run `python etl/demand_forecast.py` for the full run; here we inspect results.

In [ ]:
import demand_forecast
demand_forecast.main()
acc = pd.read_csv('../outputs/forecast_accuracy.csv')
acc

In [ ]:
res = pd.read_csv('../outputs/forecast_results.csv', parse_dates=['week'])
best_cat = acc.iloc[0]['category_name']
g = res[res.category_name == best_cat]
plt.figure(figsize=(11,4))
plt.plot(g['week'], g['actual'], label='actual', marker='o')
plt.plot(g['week'], g['predicted'], label='predicted', marker='x')
plt.title(f'Backtest — {best_cat}'); plt.legend();

## 4. Anomaly detection + ground-truth validation
Recall is measured against the injection manifest; transaction-level
precision is exact because injected keys are known.

In [ ]:
import anomaly_detector
anomaly_detector.main()
pd.read_csv('../outputs/detection_scorecard.csv')

In [ ]:
alerts = pd.read_csv('../outputs/anomaly_alerts.csv')
alerts.groupby(['severity','route_to','issue']).size().rename('alerts').reset_index()